# 00 — Setup
**Run this once before any other notebook.**

This notebook:
1. Installs all required tools and Python packages
2. Mounts Google Drive (or sets a local working directory)
3. Asks you to define your protein family (reference sequences + domain HMM)
4. Downloads your reference probes from UniProt and the HMM profile from Pfam

---
### What you need to provide
| Input | Where | Required? |
|---|---|---|
| Reference sequences | UniProt accessions **or** a FASTA file | ✅ |
| Input proteomes | A `.faa` file or NCBI taxon IDs | ✅ |
| Domain HMM | Pfam accession (auto-downloaded) **or** custom `.hmm` file | ✅ |
| Key residue motifs | Regex patterns (e.g. `H.{2}D`) | Optional |
| AlphaFold targets | UniProt accessions | Optional |

In [ ]:
# ── STEP 1: Install system tools ──────────────────────────────────────────────
!apt-get install -qq hmmer mafft fasttree

# IQ-TREE2 (ancestral reconstruction — skip if you won't use it)
import os
if not os.path.exists('/usr/local/bin/iqtree2'):
    !wget -q https://github.com/iqtree/iqtree2/releases/download/v2.3.6/iqtree-2.3.6-Linux-intel.tar.gz
    !tar -xzf iqtree-2.3.6-Linux-intel.tar.gz
    !cp iqtree-2.3.6-Linux-intel/bin/iqtree2 /usr/local/bin/iqtree2
    !rm -rf iqtree-2.3.6-Linux-intel*
    print('IQ-TREE2 installed')
else:
    print('IQ-TREE2 already installed')

In [ ]:
# ── STEP 2: Install Python packages ──────────────────────────────────────────
!pip install -q fair-esm faiss-gpu leidenalg python-igraph openTSNE \
    ete3 logomaker py3Dmol transformers accelerate biopython pyyaml seaborn

In [ ]:
# ── STEP 3: Set working directory ─────────────────────────────────────────────
import os, sys

USE_DRIVE = True   # False if running locally

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Change this to where you want results saved on your Drive:
    PROJECT_ROOT = '/content/drive/MyDrive/my_protein_survey'
else:
    PROJECT_ROOT = '/path/to/local/project'  # change to your path

os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print('Working directory:', os.getcwd())

In [ ]:
# ── STEP 4: Define your protein family ───────────────────────────────────────
#
# This is the only cell you need to customise for a new protein family.
# Fill in your details below, then run the cell.
# ─────────────────────────────────────────────────────────────────────────────

SURVEY_NAME = 'my_protein_survey'   # short name, no spaces

# Reference probes: known members of your protein family.
# Each entry: (unique_id, UniProt_accession, display_label, hex_colour)
# Use at least 2; 5–20 is ideal to cover known subfamilies.
REFERENCE_PROBES = [
    ('PROTEIN_A_HUMAN', 'P00000', 'Protein A', '#E05C5C'),
    ('PROTEIN_B_HUMAN', 'P00001', 'Protein B', '#1E88E5'),
    # add more rows...
]

# Pfam domain accession(s) for your protein family.
# Find yours at https://www.ebi.ac.uk/interpro/
# Set to None to skip HMMER prefiltering (not recommended for large datasets).
PFAM_IDS = ['PF00000']   # e.g. ['PF01553'] for AGPAT acyltransferases

# Protein length range to keep (adjust for your family).
MIN_LENGTH = 200
MAX_LENGTH = 800

# AlphaFold structures to download for comparison (optional).
# Dict of {display_name: UniProt_accession}
ALPHAFOLD_TARGETS = {
    'Protein_A': 'P00000',
    # 'Protein_B': 'P00001',
}

# Key residue motifs for structure highlighting (optional).
# Dict of {name: {pattern (regex), colour (hex), role (description)}}
# Set to {} if you don't know any yet.
CATALYTIC_MOTIFS = {
    # 'MyMotif': {'pattern': 'H.{2}D', 'colour': '#FF6B00', 'role': 'catalytic dyad'},
}

print(f'Survey: {SURVEY_NAME}')
print(f'Reference probes: {len(REFERENCE_PROBES)}')
print(f'Pfam IDs: {PFAM_IDS}')

In [ ]:
# ── STEP 5: Write config.yaml from your inputs ────────────────────────────────
import yaml

config = {
    'paths': {
        'data_dir':    'data',
        'results_dir': 'results',
    },
    'reference_probes': [
        {'id': pid, 'acc': acc, 'label': label, 'colour': colour}
        for pid, acc, label, colour in REFERENCE_PROBES
    ],
    'alphafold_targets': ALPHAFOLD_TARGETS,
    'filter':   {'min_length': MIN_LENGTH, 'max_length': MAX_LENGTH},
    'hmmer':    {'evalue': 1e-5, 'cpu': 4, 'profiles': PFAM_IDS},
    'embedding': {'model': 'esm2_t33_650M_UR50D', 'layer': 33, 'dim': 1280,
                  'batch_size': 32, 'device': 'cuda'},
    'clustering':    {'k_neighbors': 25, 'resolution': 2.0, 'pca_dims': 50, 'tsne_perp': 100},
    'subclustering': {},
    'tree':      {'mafft_flags': '--auto --thread 8 --quiet', 'fasttree_model': 'lg',
                  'iqtree_model': 'LG+G4', 'iqtree_bootstrap': 1000, 'iqtree_threads': 8},
    'structure': {'esmfold_model': 'facebook/esmfold_v1'},
    'catalytic_motifs': CATALYTIC_MOTIFS,
}

with open('config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print('config.yaml written.')
print('You can edit it directly at any time to tune clustering or tree parameters.')

In [ ]:
# ── STEP 6: Download reference probes from UniProt ────────────────────────────
from acyltransferase.config import load_config
from acyltransferase.utils  import fetch_uniprot_sequence, write_fasta

cfg = load_config('config.yaml')

probe_dir = 'data/query_sequences'
os.makedirs(probe_dir, exist_ok=True)
probes_faa = f'{probe_dir}/reference_probes.faa'

if not os.path.exists(probes_faa):
    records = []
    for probe in cfg.reference_probes:
        try:
            _, seq = fetch_uniprot_sequence(probe['acc'])
            records.append((probe['id'], seq))
            print(f"  {probe['id']}: {len(seq)} aa")
        except Exception as e:
            print(f"  {probe['id']}: FAILED — {e}")
    write_fasta(records, probes_faa)
    print(f'Saved {len(records)} probes → {probes_faa}')
else:
    from acyltransferase.utils import read_fasta
    print(f'Probes already present: {len(read_fasta(probes_faa))} sequences')

In [ ]:
# ── STEP 7: Download Pfam HMM profile ────────────────────────────────────────
from acyltransferase.search import download_hmm_profile

hmm_dir = 'data/hmm_profiles'
os.makedirs(hmm_dir, exist_ok=True)

for pfam_id in cfg.hmmer['profiles']:
    out = f'{hmm_dir}/{pfam_id}.hmm'
    if not os.path.exists(out):
        print(f'Downloading {pfam_id} ...')
        download_hmm_profile(pfam_id, out)
    else:
        print(f'{pfam_id}.hmm already present')

!hmmpress {hmm_dir}/*.hmm 2>/dev/null || true
print('Done.')

## ✓ Setup complete

**Next steps:**
1. **Upload your proteome** — put your `.faa` file at `data/proteins_raw/proteins.faa`  
   *(or provide NCBI taxon IDs and use the fetch utilities in `acyltransferase.utils`)*
2. Open **`01_embed_search.ipynb`** and set `YOUR_FASTA` to your file path
3. Run notebooks in order: `01` → `02` → `03` → `04`

---
**To change parameters later**, edit `config.yaml` directly — no need to re-run this notebook.